In [3]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# Monotonicity Function
# -------------------------------------------------
def monotonicity(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) < 2:
        return 0.0

    diff = np.diff(values)

    pos = np.sum(diff > 0)
    neg = np.sum(diff < 0)

    return abs(pos - neg) / len(diff)


# -------------------------------------------------
# Main Processing
# -------------------------------------------------
if __name__ == "__main__":

    # Load extracted features
    feature_table = pd.read_csv(r"C:\Users\psvig\OneDrive\Desktop\10_time_domain_features.csv")

    # Exclude non-feature columns, if present
    exclude_cols = ["Date", "RUL", "File Name", "Filename"]
    feature_table = feature_table.drop(
        columns=[
            col for col in exclude_cols
            if col in feature_table.columns
        ],
        errors="ignore"
    )

    # Convert columns to numeric
    feature_table = feature_table.apply(
        pd.to_numeric, errors="coerce"
    )

    # -------------------------------------------------
    # Smooth the features
    # -------------------------------------------------
    feature_table = feature_table.rolling(
        window=6, min_periods=1
    ).mean()

    # -------------------------------------------------
    # Calculate Monotonicity
    # -------------------------------------------------
    mono_scores = {}

    for col in feature_table.columns:
        mono_scores[col] = monotonicity(
            feature_table[col].values
        )

    mono_scores = pd.Series(mono_scores, name="Monotonicity")

    # -------------------------------------------------
    # Combined Feature Score
    # -------------------------------------------------
    feature_scores = pd.DataFrame({
        "Monotonicity": mono_scores
    })

    feature_scores = feature_scores.sort_values(
        by="Monotonicity", ascending=False
    )

    # -------------------------------------------------
    # Display Feature Ranking
    # -------------------------------------------------
    print("\nFeature Ranking")
    print(feature_scores)


Feature Ranking
                  Monotonicity
RMS                   0.090539
STD                   0.086470
Peak Factor           0.062055
Skewness              0.045778
Shape Factor          0.021363
RMS*Kurtosis          0.011190
Kurtosis              0.009156
Clearance Factor      0.007121
Crest Factor          0.003052
Impulse Factor        0.001017
